# HPT to Plotdaten import (import_hpt)

Pick a file from Rohdaten, pick sheets, Export to Plotdaten/.

- Engine: import_tools + template map. **Template** is Air-to-water until water-to-water / hybrid maps exist.
- Map edits: import_tools/maps/*.json

Start Jupyter with this folder as the working directory (Rohdaten/, Plotdaten/, import_tools/).

Export writes only the sheets you select. Nothing is pre-selected.

The **Prefix** is filled from the workbook name (`EXAMPLE_AW1_Lab01.xlsx` → `EXAMPLE_AW1_`, `HPT_RRT1_Lab01.xlsx` → `HPT_RRT1_`). It must end with `_` so files become `{prefix}{sheet}.xlsx`. Dry-run stays on until you uncheck it.


In [ ]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

from import_tools import list_maps, load_map, validate_plotdaten
from import_tools.detect import list_data_sheets
from import_tools.normalize import normalize_sheet, write_plotdaten

ROOT = Path('.').resolve()
ROHDATEN = ROOT / 'Rohdaten'
PLOTDATEN = ROOT / 'Plotdaten'
PLOTDATEN.mkdir(exist_ok=True)

print('cwd:', ROOT)
print('maps:', list_maps())
print('Rohdaten:', ROHDATEN, '| Plotdaten:', PLOTDATEN)

## Import UI

1. Pick an Excel under `Rohdaten/` (or paste an absolute path) → **Load sheets**.
2. **Select** the sheets to export (none selected by default).
3. Check **Prefix** (filled from the filename; edit if the profile id differs).
4. Dry-run first. If `ok` looks right, **uncheck** dry-run → **Export!**

The **map** is used when exporting (rename/units/fills). Loading sheets only lists worksheet names from the workbook.

In [ ]:
import re

META_SHEETS = {
    'Read me', 'Deliverable History', 'Lab', 'Test sequence & Settings',
    'Pictures', 'Additional data (if needed)', 'Notes', 'Test stand',
}
PREFERRED_ORDER = ['E', 'F', 'A', 'B', 'C', 'D']
_JUNK_COL = re.compile(r'^(Unnamed(\.\d+)?|Term(\.\d+)?|)$', re.I)


def list_rohdaten_xlsx():
    if not ROHDATEN.exists():
        return []
    files = [p.name for p in ROHDATEN.glob('*.xlsx') if not p.name.startswith('~$')]
    def rank(name):
        u = name.upper()
        if u.startswith('HPT') or 'AIR-TO-WATER' in u or 'HYBRID' in u or 'WATER-TO-WATER' in u:
            return (0, name)
        return (1, name)
    return sorted(files, key=rank)


def prefix_from_workbook_name(name: str) -> str:
    """EXAMPLE_AW1_Lab01.xlsx → EXAMPLE_AW1_; HPT_RRT1_Lab01.xlsx → HPT_RRT1_."""
    stem = Path(name).stem
    stem = re.sub(r'_[Ll]ab\d+$', '', stem)
    if stem and not stem.endswith('_'):
        stem += '_'
    return stem


def drop_junk_columns(df):
    if df is None or df.empty:
        return df
    drop = [c for c in df.columns if _JUNK_COL.match(str(c).strip())]
    return df.drop(columns=drop, errors='ignore') if drop else df


def _ensure_trailing_underscore(prefix: str) -> str:
    prefix = (prefix or '').strip()
    if prefix and not prefix.endswith('_'):
        prefix += '_'
    return prefix


info_box = widgets.Output(layout={'border': '1px solid #ccc', 'min_height': '120px', 'padding': '6px'})

_files = list_rohdaten_xlsx()
file_select = widgets.Select(
    options=_files,
    value=_files[0] if _files else None,
    description='Rohdaten:',
    rows=8,
    style={'description_width': '90px'},
    layout=widgets.Layout(width='480px'),
)

path_override = widgets.Text(
    value='',
    placeholder='Optional absolute path (overrides list selection)',
    description='Or path:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='640px'),
)

refresh_btn = widgets.Button(description='Refresh file list')
load_btn = widgets.Button(description='Load sheets', button_style='primary')

sheet_select = widgets.SelectMultiple(
    options=[],
    value=(),
    description='Sheets:',
    rows=8,
    style={'description_width': '90px'},
    layout=widgets.Layout(width='320px'),
)

# Label → map_id. WW / hybrid appear here when those JSON files exist.
MAP_CHOICES = [
    ('Air-to-water', 'hpt_aw_rev1'),
    ('Water-to-water', 'hpt_ww_rev1'),
    ('Hybrid', 'hpt_hybrid_rev1'),
]
_available = set(list_maps())
_map_options = [(label, mid) for label, mid in MAP_CHOICES if mid in _available]
if not _map_options:
    _map_options = [(m, m) for m in list_maps()]
map_dropdown = widgets.Dropdown(
    options=_map_options,
    value=_map_options[0][1] if _map_options else None,
    description='Template:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='360px'),
)

prefix_field = widgets.Text(
    value=prefix_from_workbook_name(_files[0]) if _files else '',
    description='Prefix:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='320px'),
)

dry_run_cb = widgets.Checkbox(
    value=True,
    description='Dry-run: no files written',
    indent=False,
)
skip_empty_cb = widgets.Checkbox(value=True, description='Skip empty sheets', indent=False)

export_btn = widgets.Button(description='Export!', button_style='success')

_state = {'raw_path': None}


def _resolve_raw_path():
    override = (path_override.value or '').strip().strip('"')
    if override:
        return Path(override)
    if not file_select.value:
        return None
    return ROHDATEN / file_select.value


def _apply_prefix_from_selection():
    raw = _resolve_raw_path()
    if raw is None:
        return
    suggested = prefix_from_workbook_name(raw.name)
    if suggested:
        prefix_field.value = suggested


def on_refresh(_=None):
    files = list_rohdaten_xlsx()
    file_select.options = files
    if files and file_select.value not in files:
        file_select.value = files[0]
    _apply_prefix_from_selection()
    with info_box:
        print(f'Refreshed: {len(files)} xlsx in Rohdaten/')
        if prefix_field.value:
            print(f'Prefix from filename: {prefix_field.value!r}')


def on_load_sheets(_):
    info_box.clear_output()
    raw = _resolve_raw_path()
    if raw is None or not raw.exists():
        with info_box:
            print('Select a Rohdaten file or set a valid absolute path.')
        return

    _apply_prefix_from_selection()
    present = list_data_sheets(str(raw), None)
    preferred = [s for s in PREFERRED_ORDER if s in present]
    extras = [s for s in present if s not in preferred and s not in META_SHEETS]
    options = preferred + extras
    sheet_select.options = options
    sheet_select.value = ()  # nothing selected by default — user must choose

    _state['raw_path'] = raw
    with info_box:
        print(f'File: {raw}')
        print(f'Sheets available: {options}')
        print(f'Prefix: {prefix_field.value!r} (from filename; edit if the profile id differs)')
        print('No sheets selected yet — select one or more, then Export.')
        print('Dry-run is ON until you uncheck it — Export will not write files.')


def on_export(_):
    info_box.clear_output()
    raw = _state.get('raw_path') or _resolve_raw_path()
    if raw is None or not Path(raw).exists():
        with info_box:
            print('No valid input file. Click Load sheets first.')
        return

    sheets = list(sheet_select.value)
    if not sheets:
        with info_box:
            print('Nothing to export: select at least one sheet (Ctrl/Shift-click).')
        return

    try:
        mapping = load_map(map_dropdown.value)
    except Exception as e:
        with info_box:
            print('Map load failed:', e)
        return

    prefix = _ensure_trailing_underscore(prefix_field.value) or prefix_from_workbook_name(Path(raw).name)
    prefix_field.value = prefix
    if not prefix:
        with info_box:
            print('Prefix is empty. Set it to the profile id plus underscore, e.g. EXAMPLE_AW1_ or HPT_RRT1_.')
        return

    dry = bool(dry_run_cb.value)
    skip_empty = bool(skip_empty_cb.value)

    with info_box:
        if dry:
            print('DRY-RUN — no files will be written. Uncheck "Dry-run: no files written" to export.')
        print(f'Export selected only: {sheets}')
        print(f"Map: {mapping.get('map_id')} {mapping.get('version')} | dry_run={dry} | skip_empty={skip_empty}")
        print(f'Input: {raw} | prefix={prefix!r}')
        print(f'Plotdaten names: {[f"{prefix}{s}.xlsx" for s in sheets]}')

    wrote = 0
    skipped = 0
    for sheet in sheets:
        df, nreport = normalize_sheet(str(raw), sheet, mapping)
        df = drop_junk_columns(df)
        vreport = validate_plotdaten(df, mapping=mapping)
        out_path = PLOTDATEN / f'{prefix}{sheet}.xlsx'
        with info_box:
            print(f"\n=== {sheet} rows={vreport['n_rows']} ok={vreport['ok']} ===")
            if nreport.get('missing_sources'):
                print('  missing required sources:', nreport['missing_sources'])
            if vreport.get('required_missing'):
                print('  missing after normalize:', vreport['required_missing'])
            if vreport.get('required_all_null'):
                print('  required all-null:', vreport['required_all_null'])
            interesting = [f for f in (nreport.get('fills') or []) if any(
                k in f for k in ('BUH', 'Virtual', 'mass', 'corrected', 'Backup', '<- 0')
            )]
            if interesting:
                print('  fills:', interesting)
            if nreport.get('warnings'):
                print('  warnings:', nreport['warnings'])

        if skip_empty and vreport['n_rows'] == 0:
            with info_box:
                print('  SKIP empty')
            skipped += 1
            continue

        if dry:
            with info_box:
                print(f'  DRY-RUN would write {out_path.name} (not written)')
                display(df.head(2))
        else:
            write_plotdaten(df, out_path)
            with info_box:
                print(f'  WROTE {out_path}')
            wrote += 1

    with info_box:
        print(f'\nDone. wrote={wrote} skipped_empty={skipped} dry_run={dry}')
        if dry:
            print('No Plotdaten files written. Uncheck dry-run and click Export! again.')
        elif wrote:
            print('Open Plotdaten files in the WebApp (Cycle Extraction / Add Data).')


def _on_file_change(change):
    if change.get('name') == 'value':
        _apply_prefix_from_selection()


file_select.observe(_on_file_change, names='value')
path_override.observe(_on_file_change, names='value')

refresh_btn.on_click(on_refresh)
load_btn.on_click(on_load_sheets)
export_btn.on_click(on_export)

display(info_box)
with info_box:
    print('1) Pick file  2) Load sheets  3) Select sheets  4) Check prefix  5) Dry-run, then uncheck dry-run to write')
    if prefix_field.value:
        print(f'Prefix from filename: {prefix_field.value!r}')
    print('Dry-run is ON — Export will not write files until you uncheck it.')

# --- file picker ---
display(widgets.HTML('<b>1. Input file</b>'))
display(widgets.HBox([file_select, refresh_btn]))
display(path_override)
display(load_btn)

# --- sheet selection ---
display(widgets.HTML('<b>2. Sheets to export</b> (multi-select; none selected by default)'))
display(sheet_select)

# --- export settings (map used here) ---
display(widgets.HTML('<b>3. Export settings</b> (map is applied when you click Export)'))
display(map_dropdown)
display(prefix_field)
display(widgets.HBox([dry_run_cb, skip_empty_cb]))
display(export_btn)


## Notes (map ≥0.4.0)

| Topic | Behaviour |
|---|---|
| Outdoor DB | `T_outdoor (DB)` only; lab air stays separate |
| Sink → WebApp | `T_out_sink`→`T_supply`, `T_in_sink`→`T_return_emu`, `volume flow_sink`→`volume flow` |
| ΔP | kPa → bar (`×0.01`) |
| Virtual Back-up | → `Electrical power input BUH` — the only series the WebApp BUH corrections read |
| Backup heater Input (optional) | → `Real BUH power input` — monitoring mean only, already inside Total Power Input |
| Both back-up series filled | Both kept, nothing merged; a warning is printed |
| No Virtual Back-up | **Omit** BUH column (do not invent zeros) |
| mass flow / corrected Q/P / either BUH | Only if present **and** non-empty in the template |
| Other optional columns (cp, density, lab, electrical, refrigerant) | Kept even when all-empty, so the field stays visible; the WebApp stores NULL and shows n/a |
| `_sink` suffix | `cp_out_sink`, `cp_in_sink`, `density_sink` keep it — water-to-water will add the `_source` twins |
| Sheets | `E, F, A, B, C, D`. `B/C/D` have no back-up headers at all, so both BUH means stay n/a |